# 38c · **Toàn bộ** từ vựng vi khuẩn (`EmbeddingBag`) trên backbone đã fine-tune

## Cái gì đổi so với `38b`

```
vi khuan   30 cot one-hot  ->  EmbeddingBag tren 304 organism + <unk>   (V = 305)
backbone   pre + ft        ->  CHI `ft` (38a/a2_E3), dong cung
```

Bỏ `pre` vì câu *"fine-tune làm EHR bớt hay thêm hữu ích"* đã trả lời ở `38b`. Còn một backbone
⇒ **~25 phút/epoch** thay vì 50.

## `29c` đo được `top-30` đã vứt đi cái gì

```
do phu LAN MOC              84.7%
study MAT SACH danh tinh     7.4%  (4,675 / 63,293)  -> chi con dem, khong con biet la con gi
study mat MOT PHAN          13.4%  (8,500 / 63,293)
<unk> (khong co trong train) 0.157%
```

Và danh sách bị cắt **không phải rác**:

```
HAEMOPHILUS INFLUENZAE      336 study
MORAXELLA CATARRHALIS       302
INFLUENZA A VIRAL ANTIGEN   238
ASPERGILLUS FUMIGATUS       235
ACINETOBACTER BAUMANNII     215
```

Tác nhân viêm phổi kinh điển — đúng nhóm nhãn mà `33` C7 chỉ ra là nhạy EHR (nhiễm trùng + chấn
thương). Nếu đuôi từ vựng có giá trị thì nó hiện ở **nhóm đó**, không ở macro. `C8` tách sẵn.

## Bag vào chuỗi như thế nào — **thêm token**, không thay token

```
EHRTok   9 ho (co ho `micro` = cac cot DEM) + 1 token BAG  ->  K = 10
EHRCol   217 cot + 1 tong hop + 1 token BAG                ->  K = 219
```

Cộng thêm, không thay thế: các cột đếm (`micro_pos_n`, `micro_n_org`, `micro_sent_*`) **ở lại** họ
`micro`; token bag mang **danh tính**. Hai thứ khác nhau, không nên nhét chung.

## Đối chứng khớp tham số vẫn phải đứng vững

`MicroBag` thêm `V × d = 305 × 256 ≈ 78k` tham số. Nếu chỉ arm thật có nó thì
`G_ehr = gate − gate_zero` sẽ **lẫn cả hiệu ứng dung lượng** — đúng cái bẫy `06b` H4z/H6z dựng ra
để tránh.

Nên arm `const` **cũng có** `MicroBag` cùng kích thước, nhưng chỉ số bị ép về hằng ⇒ nó cho ra một
**vector hằng số học được**, không đọc bệnh nhân. `C4` có cổng bắt: đưa hai bộ chỉ số khác nhau vào
bag của `gate_zero`, kết quả phải **giống hệt**.

## Ba cổng khác vẫn giữ

`gate` zero-init · `Tier1` zero-init · **tráo nhân quả** cho mọi arm có EHR — và bây giờ phép tráo
hoán vị **cả** ma trận EHR **lẫn** túi vi khuẩn, nếu không thì nó chỉ tráo một nửa.

## C0 · Config

In [ ]:
import os, json, math, time, random, shutil, tarfile, io as _io, re, glob, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'b2_prior_patch_fix'
RES38C  = os.path.join(OUT_DIR, 'stage4_nb44b', RUN_TAG)
CKPT    = os.path.join(RES38C, 'ckpt')

# LUU Y: KHONG bao gio va `builtins.open`. Da thu va hong:
#   - `_o` tra `_open` nhu bien TOAN CUC luc goi, nen chay lai cell lan 2 thi `_open` tro vao
#     chinh `_o` cu -> de quy vo han (962 tang).
#   - worker DataLoader goi `open(fd)` trong `_close_stdin` luc fork -> chet ca 8 worker.
# `ensure_res()` goi tuong minh truoc moi cum ghi la du, va no khong dung toi multiprocessing.
def ensure_res():
    '''Goi TRUOC MOI lan ghi, khong chi mot lan o C0.

       Drive FUSE danh roi thu muc RONG — Drive khong thuc su co khai niem thu muc rong nhu he
       tep thuong. `C0` tao thu muc roi KHONG ghi gi vao suot mot epoch (~23-26 phut) la du de no
       bien mat, va `np.save` o cuoi epoch bao FileNotFoundError.
       Da dinh dung loi nay o `43` (C0 ghi file thu roi XOA di -> C3 khong ghi duoc).'''
    os.makedirs(RES38C, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
    return RES38C

ensure_res()
# GHI mot file THAT va GIU lai -> thu muc khong bao gio rong nen FUSE khong danh roi
json.dump({'run_tag': RUN_TAG, 'notebook': '44b'}, open(os.path.join(RES38C, '_run_info.json'), 'w'))
open(os.path.join(CKPT, '_keep'), 'w').write('keep')   # ckpt/ cung phai co file THAT
LOCAL   = '/content/cxrlt_518'

BB_RUN = 'a2_E3'          # lan chay 38a co diem U XONG cao nhat (0.2412), KHONG lay dinh giua chung
SEEDS  = (42,)          # 1 seed — MUON bien do 0.0013 cua 44a
BACKBONES = ('ft',)       # bo `pre`: cau hoi pre-vs-ft da tra loi o 38b

W_FRONTAL, W_LATERAL = 0.7, 0.3
SEED    = 42
SIZE    = 518
EPOCHS  = 6
BATCH, ACCUM = 64, 4    # HAI anh moi mau -> ha BATCH, giu EFF 256                # EFF 240, giu nguyen qua 37 / 38a / 38b
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
G_LR_MULT = 20
NW = min(8, (os.cpu_count() or 2))

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC  = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
_ls38a = sorted(os.listdir(os.path.join(OUT_DIR, 'stage4_nb38a'))) \
         if os.path.isdir(os.path.join(OUT_DIR, 'stage4_nb38a')) else []
assert os.path.exists(BB_SRC),  f'khong thay {BB_SRC}. 38a co san: {_ls38a}'
assert os.path.exists(P_MICRO), f'khong thay {P_MICRO} -> chay 29c_cxrlt_micro_full.ipynb truoc'
assert os.path.exists(P_VOCAB), f'khong thay {P_VOCAB} -> chay 29c truoc'

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head', 'medium', 'tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id', 'view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL          # CUNG cach cat nhu 35/37/38a/38b
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA = np.where(TF_)[0], np.where(SL)[0], np.where(VA)[0]
N_POS_TF = YM[TF_].sum(0)

print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'ghi vao: {RES38C}')
print(f'backbone `ft` = 38a/{BB_RUN} | seeds {SEEDS} | E={EPOCHS} | BATCH {BATCH}x{ACCUM}')

Mounted at /content/drive
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
377,110 anh | khop 219,785 | chon 39,086 | val 39,293
ghi vao: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb44b/b2_prior_patch_fix
backbone `ft` = 38a/a2_E3 | seeds (42,) | E=6 | BATCH 64x4


## C1 · EHR — **bỏ 30 cột one-hot**, nạp túi vi khuẩn đầy đủ

Phần trên verbatim `37`. Phần dưới làm đúng hai việc:

**1 · Bỏ mọi cột `micro_org_*` khỏi ma trận đặc.** Nếu để lại thì danh tính bị **đếm hai lần** —
một lần qua cột, một lần qua bag. Đó là loại lỗi **im lặng**, không `assert` nào bắt được về sau,
nên nó phải bị bắt ở đây, và số cột bỏ đi phải **khớp đúng 30**.

**2 · Nạp túi ở dạng ĐỆM `[N, L]` thay vì `offsets`.** Túi rất ngắn (trung bình `106,114/63,293 ≈
1.7` vi khuẩn mỗi study) nên đệm rẻ hơn nhiều so với viết `collate_fn` riêng — và
`nn.EmbeddingBag` nhận thẳng đầu vào 2 chiều. Ô đệm mang **trọng số 0** ⇒ đóng góp 0 vào tổng
**và** gradient 0 về hàng đó (`∂out/∂E[i] = w_i = 0`), nên không rò.

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()

def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = {'Q_CTX': [c for c in SPEC['Q_CTX'] if c in E_DF.columns],
        'Q_FULL': [c for c in SPEC['Q_FULL'] if c in E_DF.columns]}
EHR = {k: _mat(v) for k, v in COLS.items()}
# chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val
EHR_MU, EHR_SD = {}, {}
EHR_CLIP = 10.0     # cot NHI PHAN cuc hiem lam chuan hoa no ra gia tri KHONG LO:
                    #   1 duong / 219,785  ->  sd 2.1e-3  ->  gia tri chuan hoa 468
                    # Probe tuyen tinh (35) chiu duoc — chi can hoc trong so nho. ATTENTION thi
                    # KHONG: `EHRColTok` dua thang x_j vao token (T_j = b_j + x_j*W_j), va score
                    # ti le voi M^2 nen bien do 30 cho score ~2e4 -> softmax bao hoa -> inf/NaN.
BIN_OF = {}
for k, m in EHR.items():
    # Cot NHI PHAN thi KHONG z-score. z-score bien gia tri "co" thanh sqrt((1-p)/p):
    #   p = 1/219,785 -> 468   |   p = 0.01 -> 9.9   |   p = 0.5 -> 1.0
    # Do chinh la nguon cua gia tri khong lo, va no la LOI BIEU DIEN chu khong phai
    # loi cua du lieu: cot chi bao khong co "do lech chuan" de chia.
    # `29b` one-hot top-K vi khuan (`micro_org_*`) + co comorbidity => rat nhieu cot cuc hiem.
    b_ = np.array([np.isin(np.unique(m[:, j]), (0.0, 1.0)).all() for j in range(m.shape[1])])
    BIN_OF[k] = b_
    mu = m[TF_].mean(0, keepdims=True); sd = m[TF_].std(0, keepdims=True) + 1e-6
    mu[:, b_] = 0.0; sd[:, b_] = 1.0                       # nhi phan: giu nguyen 0/1
    EHR_MU[k], EHR_SD[k] = mu, sd
    z = (m - mu) / sd
    _n_out = int((np.abs(z) > EHR_CLIP).sum())
    EHR[k] = np.clip(z, -EHR_CLIP, EHR_CLIP).astype('float32')  # kep chi con la chan cuoi
    print(f'  {k}: {int(b_.sum())}/{len(b_)} cot NHI PHAN (giu 0/1) | '
          f'|z| max {np.abs(z).max():6.1f} | kep {_n_out:,} o ({_n_out/z.size:.4%})')

# --- gom cot thanh 8 HO. Quy tac, khong phai tay; moi cot phai roi vao DUNG mot ho. ---
FAM_RE = [('ctx',    r'^ctx_'),
          ('demo',   r'^(age|gender)$'),   # nhan khau hoc: LUON co mat, khong phai phep do.
                                           # `29b` xep vao "gia tri" vi muc dich cua no la hoi
                                           # "loai benh nhan vs benh"; o day chung la HO rieng.
          ('mask',   r'_mask$'),
          ('n_lab',  r'^n_lab_'),
          ('dh_lab', r'^dh_lab_'),
          ('lab',    r'^lab_'),
          ('vital',  r'^vital_'),
          ('como',   r'^como_'),
          ('micro',  r'^micro_')]
def famsplit(cols):
    out, seen = {}, set()
    for nm, rx in FAM_RE:
        g = [i for i, c in enumerate(cols) if re.search(rx, c) and i not in seen]
        seen |= set(g)
        if g: out[nm] = np.array(g)
    du = [c for i, c in enumerate(cols) if i not in seen]
    assert not du, f'cot khong thuoc ho nao: {du[:10]}'
    return out
FAM = {k: famsplit(v) for k, v in COLS.items()}
for k in COLS:
    print(f'{k}: {len(COLS[k])} cot -> {len(FAM[k])} ho  ' +
          ' | '.join(f'{n}:{len(ix)}' for n, ix in FAM[k].items()))

# ================== 38c: bo cot one-hot, nap tui vi khuan day du ==================
_MICRO_RE = re.compile(r'^micro_org_')
_COLS0  = {k: list(v) for k, v in COLS.items()}                       # truoc khi bo
_KEEP   = {k: [i for i, c in enumerate(v) if not _MICRO_RE.match(c)] for k, v in _COLS0.items()}
_n_drop = {k: len(_COLS0[k]) - len(_KEEP[k]) for k in _COLS0}
print('')
print(f'bo cot one-hot micro_org_*: {_n_drop}  (danh tinh gio di qua BAG)')
assert _n_drop['Q_FULL'] == 30, (
    f"Q_FULL bo {_n_drop['Q_FULL']} cot micro_org_*, phai la 30. "
    '29b da doi ORG_TOP_K? DUNG chay tiep — bag va cot se dem HAI LAN.')
COLS   = {k: [_COLS0[k][i] for i in _KEEP[k]] for k in _COLS0}
EHR    = {k: EHR[k][:, _KEEP[k]]    for k in _COLS0}
EHR_MU = {k: EHR_MU[k][:, _KEEP[k]] for k in _COLS0}
EHR_SD = {k: EHR_SD[k][:, _KEEP[k]] for k in _COLS0}
BIN_OF = {k: BIN_OF[k][_KEEP[k]]    for k in _COLS0}
FAM    = {k: famsplit(v) for k, v in COLS.items()}
for k in COLS:
    assert EHR[k].shape[1] == len(COLS[k]), f'{k}: ma tran {EHR[k].shape[1]} vs {len(COLS[k])} cot'
    print(f'  {k}: {len(COLS[k])} cot -> {len(FAM[k])} ho  ' +
          ' | '.join(f'{n}:{len(ix)}' for n, ix in FAM[k].items()))
assert 'micro' in FAM['Q_FULL'], 'ho `micro` bien mat -> da bo ca cot DEM chu khong chi one-hot'

# --- tui vi khuan: parquet ragged -> mang DEM [N, L] ---
_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1        # +1 hang PAD o cuoi
PAD_ID = V_MIC - 1
_mic = _MIC.reindex(ORDER.study_id.to_numpy())
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max()))
assert MIC_L <= 64, f'tui dai nhat {MIC_L} vi khuan -> dem se ton bo nho, xem lai 29c'
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64')
MCNT = np.zeros((len(ORDER), MIC_L), 'float32')          # o dem trong so 0 => dong gop 0, grad 0
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b
_have = (MCNT > 0).any(1)
print(f'\ntui vi khuan: V={V_MIC-1} (+1 PAD) | dai toi da {MIC_L} | '
      f'anh co it nhat 1 vi khuan {_have.sum():,}/{len(ORDER):,} ({_have.mean():.1%})')
assert (MIDS[MCNT == 0] == PAD_ID).all() and (MIDS[MCNT > 0] != PAD_ID).all(), \
    'o dem va o that lan nhau'
assert MIDS.max() < V_MIC and MIDS.min() >= 0
print('cong: khong con cot one-hot · ma tran khop so cot · o dem tach bach')

# ================== PHIM CHUP TRUOC: hang ORDER cua study gan nhat truoc do ==================
_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_hv = set(ORDER.study_id.unique())
_B = _BASE[_BASE.study_id.isin(_hv)].sort_values(['subject_id','index_time']).reset_index(drop=True)
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _sid in enumerate(ORDER.study_id.to_numpy()): _rw.setdefault(_sid, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _sid in enumerate(ORDER.study_id.to_numpy()):
    _p2 = _PV.get(_sid, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hasp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hasp.sum():,} / {len(ORDER):,} ({_hasp.mean():.1%})  (44a do 67.2%)')
assert _hasp.mean() > 0.5, 'ty le qua thap -> kiem lai ghep study'
for _nm, _m in (('khop', TF_), ('chon', SL), ('val', VA)):
    print(f'  {_nm:>4}: {(_hasp & _m).mean():.1%}')
assert (PREV_ROW[_hasp] != np.arange(len(ORDER))[_hasp]).all(), 'phim truoc TRUNG chinh no'

  Q_CTX: 47/157 cot NHI PHAN (giu 0/1) | |z| max   88.2 | kep 14,968 o (0.0253%)
  Q_FULL: 58/247 cot NHI PHAN (giu 0/1) | |z| max  138.4 | kep 50,331 o (0.0540%)
Q_CTX: 157 cot -> 5 ho  ctx:2 | mask:45 | n_lab:54 | dh_lab:54 | vital:2
Q_FULL: 247 cot -> 9 ho  ctx:2 | demo:2 | mask:45 | n_lab:54 | dh_lab:54 | lab:36 | vital:8 | como:11 | micro:35

bo cot one-hot micro_org_*: {'Q_CTX': 0, 'Q_FULL': 30}  (danh tinh gio di qua BAG)
  Q_CTX: 157 cot -> 5 ho  ctx:2 | mask:45 | n_lab:54 | dh_lab:54 | vital:2
  Q_FULL: 217 cot -> 9 ho  ctx:2 | demo:2 | mask:45 | n_lab:54 | dh_lab:54 | lab:36 | vital:8 | como:11 | micro:5

tui vi khuan: V=305 (+1 PAD) | dai toi da 10 | anh co it nhat 1 vi khuan 88,674/377,110 (23.5%)
cong: khong con cot one-hot · ma tran khop so cot · o dem tach bach
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)
  khop: 38.9%
  chon: 6.9%
   val: 7.0%


## C2 · Ảnh local + backbone `ft` đông cứng

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n_local = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n_local:,} / {len(ORDER):,}')
if _n_local < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        name, blob = pl
        dst = os.path.join(LOCAL, os.path.basename(name))
        if os.path.exists(dst): return 0
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95); return 1
    t0 = time.time()
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n_local = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'stage xong: {_n_local:,} anh | {(time.time()-t0)/60:.1f} phut')
assert _n_local >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n_local:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_pre = AutoModel.from_pretrained('microsoft/rad-dino')
_sd0 = {k: v.clone() for k, v in _pre.state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
assert _d > 1e-6, f'backbone GIONG HET ban pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
del _pre, _sd0
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'BACKBONE `ft`: 38a/{BB_RUN}  E={_ck.get("E")}, epoch cuoi={_ck.get("ep")}, '
      f'ft-lp={_ck.get("ft_lp", float("nan")):+.4f} | lech so voi pretrain {_d:.3f}')
print('  CHI lay `bb`. `head` trong file KHONG dung — moi arm khoi tao ngau nhien cung seed.')

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ co `has` bao co phim truoc khong).
       `ehr_perm`: hoan vi hang EHR cho phep TRAO NHAN QUA. Nhan y KHONG di theo.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i])
        x = s._img(i)
        xp = s._img(pr) if pr >= 0 else torch.zeros_like(x)   # khong co phim truoc -> anh 0, has=0
        j = i if s.perm is None else int(s.perm[k])
        return dict(image=x, image_prev=xp, has=torch.tensor(1.0 if pr >= 0 else 0.0),
                    sub=int(SUB[i]), pos=k,   # `pos` de ghi du doan khi loader SHUFFLE
                    y=torch.from_numpy(YM[i]), row=i, erow=j)

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def fwd(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone(pixel_values=img).last_hidden_state[:, 1:].float()

_b = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
N_TOK = fwd(_b['image'].to(dev)).shape[1]
print(f'patch {N_TOK} token')
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2

anh local: 0 / 377,110


stage:   0%|          | 0/378 [00:00<?, ?tar/s]

stage xong: 377,110 anh | 23.1 phut


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft`: 38a/a2_E3  E=3, epoch cuoi=3, ft-lp=+0.0076 | lech so voi pretrain 0.021
  CHI lay `bb`. `head` trong file KHONG dung — moi arm khoi tao ngau nhien cung seed.
patch 1369 token


## C3 · Kiến trúc — verbatim `37`, cộng `MicroBag`

`MicroBag` **không phải kiến trúc mới**. `Linear` áp lên vector one-hot có đếm **bằng đúng**
`EmbeddingBag(mode='sum', per_sample_weights=đếm)` — đã kiểm offline tới `2.98e-08`. Thay đổi duy
nhất là **từ vựng**: `30 → 305`, và chi phí `O(số vi khuẩn có mặt)` thay vì `O(V)`.

In [ ]:
class GroupFC(nn.Module):
    '''Moi lop mot vector trong so rieng -> 1 logit. Khong chia se giua cac lop.'''
    def __init__(s, C, d):
        super().__init__(); s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b        # [B, C, d] -> [B, C]

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a)
        return s.n2(h + s.ff(h)), w

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = chuoi token.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, ehr='none', d_ehr_in=0, K=8,
                 t1=False):
        super().__init__()
        s.mode = ehr; s.K = K
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d)
        s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None      # B2a — dat TRUOC tang 2
        if ehr != 'none' or t1:
            s.ein = nn.Linear(d_ehr_in, d)
            if ehr == 'gate':
                s.ehr = XAttn(d, nhead, ff, drop)
                s.pe  = nn.Linear(d, d)                # KHOI TAO BINH THUONG — xem ghi chu duoi
                s.g = nn.Parameter(torch.zeros(C))     # cong TUNG LOP, init 0 (mau ReZero)
                # KHONG zero-init CA HAI. `h + g*pe(he)` voi g=0 VA pe=0 la diem yen ngua chet:
                #   dL/dg  = pe(he) = 0   va   dL/dpe = g = 0   -> nhanh EHR khong bao gio chay.
                # Dry-run da bat dung loi nay (g_c khong nhuc nhich sau 3 epoch).
                # Chi `g=0` la du de khoi dong DUNG BANG CXR-only, va khi do dL/dg = pe(he) != 0.
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        B = patch.shape[0]
        q  = s.q.weight.unsqueeze(0).expand(B, -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None:                              # TANG 1: patch duoc EHR dieu bien
            kv = s.t1(kv, s.ein(etok))                    # theo VUNG, truoc khi query lop doc
        if s.mode == 'cat':
            kv = torch.cat([kv, s.ein(etok)], 1)                 # softmax tren 1369+K
            h, w = s.img(q, kv, need_w); return s.out(h), w
        h, w = s.img(q, kv, need_w)
        if s.mode == 'gate':
            he, we = s.ehr(q, s.ein(etok), need_w)               # softmax CHI tren K token EHR
            h = h + s.g.view(1, -1, 1) * s.pe(he)
            return s.out(h), (w, we)
        return s.out(h), w

class EHRTok(nn.Module):
    '''K token EHR. `fam`: dict ho -> chi so cot. mode `zero` = vector HANG SO hoc duoc,
       khop tham so voi ban that (doi chung H4z/H6z cua 06b).'''
    def __init__(s, fam, d=256, const=False):
        super().__init__()
        s.fam, s.const = fam, const
        s.enc = nn.ModuleList([nn.Sequential(nn.Linear(len(ix), d), nn.GELU(), nn.Linear(d, d))
                               for ix in fam.values()])
        s.idx = [torch.as_tensor(ix) for ix in fam.values()]
        if const: s.c = nn.Parameter(torch.randn(len(fam), d) * 0.02)
        s.d = d
    def forward(s, e):
        if s.const: return s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        return torch.stack([m(e[:, ix.to(e.device)]) for m, ix in zip(s.enc, s.idx)], 1)

class EHRColTok(nn.Module):
    '''247 token COT (FT-Transformer: T_j = b_j + x_j*W_j) + 1 token TONG HOP tu MLP nhin
       CA 247 cot cung luc.

       Tinh chat quyet dinh: chuoi nay CHUA k1 nhu TRUONG HOP RIENG — attention chi can don
       het khoi luong len token tong hop la tai lap duoc k1. `EHRTok` theo HO (K=9) KHONG co
       tinh chat do (khong ho nao chua ca 247 cot), va do chinh la ly do no THUA k1 o v1.

       Gop dung hai thu ma K=1 va K=9 moi ben chi co mot:
         token cot     -> CHON LOC tung cot ("Pneumonia doc cot bach cau", khong phai ca ho)
         token tong hop-> TUONG TAC CHEO (attention la to hop LOI, khong tao duoc tich cot)

       HUONG A: moi cot LUON co token. Cot thieu -> (0-mu)/sigma, mot DIEM CO DINH tren duong
       thang cua cot do, mo hinh cham diem thap cho no duoc. Khong `key_padding_mask` => khong
       co kieu loi 'hang toan -inf -> NaN' voi 11,839 study khong co ban ghi nao.'''
    def __init__(s, d_in, d=256, const=False):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d)   # chan bien do token: `col` dua thang x_j vao token nen mot cot
                                 # hiem co the lan at 247 token con lai. `EHRTok` khong can vi
                                 # gia tri cua no da di qua GELU.
        s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)   # doi chung: HANG SO hoc duoc
    def forward(s, e):
        if s.const: return s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)     # [B, 247, d]
        return s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))       # [B, 248, d]

class Tier1(nn.Module):
    '''B2a — Q = 1369 PATCH, K/V = token EHR. FiLM BIEN THIEN THEO KHONG GIAN.

       softmax chi tren token EHR nen cac patch KHONG canh tranh voi nhau: moi patch tu hoi
       EHR rieng, va EHR khong the bi 1369 patch nhan chim (do la co che that bai cua 06b H6).
       Tra loi 'boi canh khien VUNG NAO dang ngo' — truc KHONG GIAN, khac han truc NGU NGHIA
       cua tang 2 ('NHAN NAY co quan tam EHR khong').

       `pr` ZERO-INIT => khoi dong DUNG BANG CXR-only. CHI MOT he so zero-init: khac v1, noi
       `g_c=0` VA `pe=0` tao diem yen ngua chet (dL/dg = pe(he) = 0 va dL/dpe ~ g = 0).
       O day dL/dpr = dL/dpatch' != 0 ngay tu buoc dau.

       Chi phi: 1369 x 248 attention = 18% mot lop self-attention (1369^2 = 1.87M).
       `need_weights=False` cho phep duong SDPA hop nhat -> khong hien thuc hoa ma tran [1369, 248].'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n  = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
        # BAN VA 38d: chuan hoa DAU VAO cua `pr`, KHONG phai dau ra.
        #   38c: `patch + pr(a)` khong co gi chan. Voi token HANG SO, `etok` giong het moi mau nen
        #   `a` chi con phu thuoc `patch` -> tu-anh-xa tu do trong nhanh residual -> phinh ra va
        #   nuot bieu dien anh (`t1_zero` 0.2416 -> 0.0681 o ep3 cua 38c).
        #   KHONG LayerNorm SAU phep cong: LN cua vector be xiu van cho bien do O(1), pha tinh
        #   chat zero-init. Chuan hoa dau vao thi `pr` zero-init van cho DUNG 0, va bien do nhieu
        #   loan ti le voi ||pr|| — thu ma weight decay kiem soat duoc.
        s.ln_a = nn.LayerNorm(d)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class MeanLin(nn.Module):
    def __init__(s, C, d_in, d_ehr=0):
        super().__init__(); s.fc = nn.Linear(d_in + d_ehr, C); s.use_e = d_ehr > 0
    def forward(s, patch, e=None):
        h = patch.mean(1)
        return s.fc(torch.cat([h, e], 1) if s.use_e else h), None

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

# ================== 38c: MicroBag + hai lop token co bag ==================
class ConstLN(nn.Module):
    '''Boc mot lop token EHR de duong HANG SO cung bi chan bien do nhu duong THAT.

       Loi da lam chet mot lan chay: `EHRTok`/`EHRColTok` o che do `const` tra thang `s.c`,
       KHONG qua LayerNorm/GELU, trong khi duong that thi co. `s.c` la tham so tu do, khong
       gi chan, lon dan qua cac epoch -> diem attention (ti le M^2) tran -> NaN.
       Trieu chung: `mldec_t1` song, `mldec_t1_zero` chet o ep3.

       Chan bang chinh LayerNorm ma duong that dung => hai nhanh cung thang do, va `G_ehr`
       van la doi chung khop tham so (+2d tham so, khong dang ke).'''
    def __init__(s, inner, d, const):
        super().__init__(); s.inner, s.const = inner, const
        s.ln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, *a, **k): return s.ln(s.inner(*a, **k))

class MicroBag(nn.Module):
    '''EmbeddingBag tren TOAN BO tu vung vi khuan.
       `const=True`: doi chung khop tham so — GIU nguyen bang nhung ep moi chi so ve 0 va trong so
       ve 1, nen no cho ra mot VECTOR HANG SO hoc duoc, khong doc benh nhan (mau H4z/H6z cua 06b).
       Khong lam vay thi arm that co them V*d tham so ma doi chung khong co, va G_ehr se lan ca
       hieu ung DUNG LUONG — dung cai bay ma doi chung sinh ra de tranh.'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.bag = nn.EmbeddingBag(V, d, mode='sum')
        nn.init.normal_(s.bag.weight, std=d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))    # khop dang cua EHRTok
        s.const = const
    def forward(s, ids, w):
        if s.const:
            # MOT chi so, khong phai MIC_L ban sao: tui that trung binh 1.7 vi khuan, nen
            # `MIC_L x E[0]` lam doi chung co bien do lon gap ~10x ban that va khuech dai
            # gradient ve hang 0 dung bay nhieu lan.
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w   = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(s.bag(ids, per_sample_weights=w))       # [B, d]

class EHRTokBag(EHRTok):
    '''9 ho + 1 token BAG. Cot DEM (`micro_pos_n`, `micro_n_org`, ...) o lai ho `micro`;
       token bag mang DANH TINH. Hai thu khac nhau, khong nhet chung.'''
    def __init__(s, fam, d=256, const=False, V=None):
        super().__init__(fam, d, const)
        s.mb = MicroBag(V, d, const); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids, mcnt):
        return torch.cat([s.cln(super().forward(e)), s.mb(mids, mcnt).unsqueeze(1)], 1)

class EHRColTokBag(EHRColTok):
    '''217 cot + 1 tong hop + 1 token BAG. LayerNorm rieng cho token bag de khop bien do
       voi phan con lai (EHRColTok da LN cho col+joint).'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__(d_in, d, const)
        s.mb = MicroBag(V, d, const); s.bln = nn.LayerNorm(d)
        s.cln = nn.LayerNorm(d) if const else nn.Identity()   # duong that da co s.ln
    def forward(s, e, mids, mcnt):
        return torch.cat([s.cln(super().forward(e)),
                          s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1Img(nn.Module):
    '''Phim chup TRUOC o muc PATCH. Q = 1369 patch HIEN TAI, K/V = 1369 patch PHIM CU.
       Chay o khong gian D_CXR (TRUOC `pin` cua MLDecoder) => MLDecoder khong phai sua gi.

       Vi sao KHONG tru thang `patch_cur - patch_prior`: token `i` ung voi mot O LUOI cu the, ma
       hai phim KHONG duoc dang ky (tu the, do hit vao, xoay khac). O `i` co the la dinh phoi o
       phim nay va xuong don o phim kia -> hieu bi LECH DANG KY chi phoi.
       Cross-attention tu tim tuong ung: `a[i]` la to hop patch cu KHOP voi patch hien tai `i`.
       (O `43`/`44a` tru duoc vi `mean(patch)` da XOA SACH toa do — khong con chi so de lech.)

       diff=True  -> DONG CUNG phep hieu SAU khi da can:  pr(ln_a(patch - a))
       diff=False -> dua phim cu DA CAN, mo hinh tu quyet: pr(ln_a(a))
       Khong thay the nhau: `ln_a` phi tuyen nen ln_a(p-a) != ln_a(p) - ln_a(a). Chay CA HAI.

       `pr` zero-init + `ln_a` chuan hoa DAU VAO — ban va cua 38d (`t1_zero` da sup vi thieu no).'''
    def __init__(s, d, nhead=8, drop=0.1, diff=False):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d); nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.pr(s.ln_a(patch - a if s.diff else a))
        # anh KHONG co phim truoc -> mask nhanh di han. Khong bia phim gia, va khong lay chinh no
        # lam phim truoc (the la self-attention = them dung luong tra hinh).
        return patch + z * has.view(-1, 1, 1)

print('kien truc san sang (co MicroBag + ConstLN + Tier1Img)')

kien truc san sang
kien truc san sang (co MicroBag + ConstLN + Tier1Img)


## C4 · **Ba** arm, và cổng cho `Tier1`

`38c` chạy 12 arm; ở đây chỉ giữ đúng bộ ba cần cho một câu hỏi:

```
mldec           goc toa do, khong EHR
mldec_t1        Tier1 that   -> `t1 - mldec` = +0.0071 o 38c (bien do 0.0002)
mldec_t1_zero   Tier1 HANG SO, khop tham so
                -> `t1 - t1_zero` tach phan EHR khoi phan DUNG LUONG cua Tier1
```

Ở `38c`, `t1_zero` **sụp** (`0.2416 → 0.0681`) nên phép tách đó không đọc được — và `t1` lại đúng là
arm dẫn đầu. Đây là arm duy nhất thiếu đối chứng.

**Cổng mới (`cong 8`):** chạy `Tier1` với token hằng số **rồi phóng đại đầu vào 100×** — biên độ
đầu ra phải **gần như không đổi**, vì `ln_a` chuẩn hoá trước phép chiếu. `38c` sẽ trượt cổng này.

In [ ]:
D_MLD, D_ETOK = 512, 256
ARMS = {
  'mldec':      dict(head='mld', ehr=None, mode='none'),
  'prior_a':    dict(head='mld', ehr=None, mode='none', img=True),                # phim cu DA CAN
  'prior_d':    dict(head='mld', ehr=None, mode='none', img=True, diff=True),     # HIEU sau khi can
  'prior_zero': dict(head='mld', ehr=None, mode='none', img=True, diff=True, shuf=True),  # DOI CHUNG
  'mldec_t1':   dict(head='mld', ehr='Q_FULL', mode='none', tok='col', t1=True),  # moc EHR +0.0056
  't1_prior':   dict(head='mld', ehr='Q_FULL', mode='none', tok='col', t1=True, img=True, diff=True),
}
IMG = nn.ModuleDict()      # nhanh PHIM TRUOC, tach rieng khoi MOD de MLDecoder khong phai sua

def build_arms(seed=SEED):
    torch.manual_seed(seed)
    M, E, I = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        K = 0
        if cfg['ehr']:
            n_in, cst = EHR[cfg['ehr']].shape[1], cfg.get('const', False)
            E[tag] = EHRColTokBag(n_in, D_ETOK, cst, V=V_MIC); K = n_in + 2   # +1 joint +1 bag
        M[tag] = MLDecoder(C, D_CXR, D_MLD, ehr=cfg['mode'], d_ehr_in=D_ETOK, K=K,
                           t1=cfg.get('t1', False))
        if cfg.get('img'):
            I[tag] = Tier1Img(D_CXR, diff=cfg.get('diff', False))
    return M.to(dev), E.to(dev), I.to(dev)

MOD, ETK, IMG = build_arms()
print(f'{"arm":<17} {"tham so":>12}')
for tag in ARMS:
    n = sum(p.numel() for p in MOD[tag].parameters() if p.requires_grad)
    n += sum(p.numel() for p in ETK[tag].parameters()) if tag in ETK else 0
    n += sum(p.numel() for p in IMG[tag].parameters()) if tag in IMG else 0
    print(f'  {tag:<15} {n:>12,}')

_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev)
_mc = torch.zeros(4, MIC_L, device=dev)
_p = torch.randn(4, N_TOK, D_CXR, device=dev)
_e = torch.randn(4, EHR['Q_FULL'].shape[1], device=dev)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'

# --- cong 5: Tier1 zero-init => bo tang 1 phai cho logit Y HET ---
with torch.no_grad():
    _t = ETK['mldec_t1'](_e, _mi, _mc); MOD['mldec_t1'].eval()
    l3, _ = MOD['mldec_t1'](_p, _t)
    _keep = MOD['mldec_t1'].t1; MOD['mldec_t1'].t1 = None
    l4, _ = MOD['mldec_t1'](_p, _t)
    MOD['mldec_t1'].t1 = _keep
    d1 = (l3 - l4).abs().max().item()
assert d1 < 1e-5, f'Tier1 KHONG zero-init: lech {d1:.2e}'
print(f'\ncong: backbone dong cung OK | query co dinh OK | Tier1 zero-init OK (lech {d1:.1e})')

# --- cong 6: chuoi col = 217 cot + 1 tong hop + 1 bag ---
with torch.no_grad():
    _nc = ETK['mldec_t1'](_e, _mi, _mc).shape[1]
assert _nc == EHR['Q_FULL'].shape[1] + 2, f'chuoi col = {_nc}, phai la {EHR["Q_FULL"].shape[1]}+2'
print(f'cong: chuoi col = {_nc} token ({_nc-2} cot + 1 tong hop + 1 BAG vi khuan) OK')

_pp = torch.randn(4, N_TOK, D_CXR, device=dev)     # "phim truoc" gia
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)

# --- cong 8: `ln_a` PHAI chan duoc bien do dau vao phep chieu (ban va 38d) ---
with torch.no_grad():
    _ti = IMG['prior_a']; _ti.pr.weight.normal_(0, 0.02)
    _r1 = (_ti(_p, _pp, _h1) - _p).abs().max()
    _r2 = (_ti(_p, _pp * 100.0, _h1) - _p).abs().max()     # PHONG DAI K/V 100 lan
    _r = float(_r2 / (_r1 + 1e-9)); nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, (f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc. '
                  'Day dung la loi da lam `t1_zero` sup o 38c.')
print(f'cong 8: phong dai K/V 100x -> bien do ra chi doi {_r:.2f}x (ban va `ln_a` OK)')

# --- cong 9: Tier1Img zero-init => khoi dong DUNG BANG khong co phim truoc ---
with torch.no_grad():
    for _t in IMG:
        assert (IMG[_t](_p, _pp, _h1) - _p).abs().max() < 1e-6, f'{_t}: Tier1Img KHONG zero-init'
print(f'cong 9: Tier1Img zero-init OK cho {list(IMG)} — khoi dong DUNG BANG arm khong co phim truoc')

# --- cong 10: has=0 PHAI cat sach nhanh, ke ca khi `pr` da khac 0 ---
with torch.no_grad():
    _ti = IMG['prior_d']; _ti.pr.weight.normal_(0, 0.05)
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
print(f'cong 10: has=0 cat sach nhanh OK (32.8% anh khong co phim truoc)')

# --- cong 11: `diff` PHAI doi ket qua (neu khong, hai arm prior_a/prior_d la ban sao) ---
with torch.no_grad():
    IMG['prior_a'].pr.weight.normal_(0, 0.05)
    IMG['prior_d'].pr.weight.copy_(IMG['prior_a'].pr.weight)
    IMG['prior_d'].nq.load_state_dict(IMG['prior_a'].nq.state_dict())
    IMG['prior_d'].nk.load_state_dict(IMG['prior_a'].nk.state_dict())
    IMG['prior_d'].at.load_state_dict(IMG['prior_a'].at.state_dict())
    _da = (IMG['prior_a'](_p, _pp, _h1) - IMG['prior_d'](_p, _pp, _h1)).abs().max().item()
    for _t in ('prior_a', 'prior_d'): nn.init.zeros_(IMG[_t].pr.weight)
assert _da > 1e-4, f'`diff=True/False` cho ket qua nhu nhau (lech {_da:.2e}) -> hai arm la ban sao'
print(f'cong 11: diff=True khac diff=False (lech {_da:.3f}) — hai arm THUC SU khac nhau')

arm                    tham so
  mldec              2,517,032
  prior_a            5,474,600
  prior_d            5,474,600
  prior_zero         5,474,600
  mldec_t1           4,341,800
  t1_prior           7,299,368

cong: backbone dong cung OK | query co dinh OK | Tier1 zero-init OK (lech 0.0e+00)
cong: chuoi col = 219 token (217 cot + 1 tong hop + 1 BAG vi khuan) OK
cong 8: phong dai K/V 100x -> bien do ra chi doi 1.04x (ban va `ln_a` OK)
cong 9: Tier1Img zero-init OK cho ['prior_a', 'prior_d', 'prior_zero', 't1_prior'] — khoi dong DUNG BANG arm khong co phim truoc
cong 10: has=0 cat sach nhanh OK (32.8% anh khong co phim truoc)
cong 11: diff=True khac diff=False (lech 10.504) — hai arm THUC SU khac nhau


## C5 · Chấm điểm — verbatim `32` C2 / `35` C3 / `37` C6 / `38b` C5

In [ ]:
from sklearn.metrics import average_precision_score
_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V_.view_group.to_numpy()[VA] == 'frontal') & V_.use.to_numpy()[VA]
_isL = (V_.view_group.to_numpy()[VA] == 'lateral') & V_.use.to_numpy()[VA]

def pool_study(p):
    out = np.zeros((len(_uid), p.shape[1]), 'float64'); ws = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(p.shape[1])], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; ws[has] += w
    ok = ws > 0; out[ok] /= ws[ok, None]; return out, ok
_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Ys = YM[VA][_first]

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def score(p):
    ap_i = ap_cols(YM[VA], p)
    ps, ok = pool_study(p)
    ap_s = np.array([average_precision_score(_Ys[ok][:, j], ps[ok][:, j]) for j in range(C)])
    return ap_i, ap_s
print(f'val {VA.sum():,} anh -> {len(_uid):,} study')

val 39,293 anh -> 23,800 study


## C6 · Vòng lặp — mỗi seed một lần chạy đầy đủ

Không early-stop, cosine trải trên `E`, lấy **epoch cuối**. Clip **từng arm**. `RESUME` theo
`(seed, epoch)`.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T = torch.from_numpy(MIDS); MCNT_T = torch.from_numpy(MCNT)

def ehr_batch(M, E, tag, erow):
    cfg = ARMS[tag]
    if not cfg['ehr']: return None
    e = torch.from_numpy(EHR[cfg['ehr']][erow.numpy()]).to(dev)
    if cfg['head'] == 'mean': return e            # mean_lin: chi ma tran dac, khong co bag
    return E[tag](e, MIDS_T[erow].to(dev), MCNT_T[erow].to(dev))

SHUF_SAME = []      # ty le doi chung TRUNG benh nhan — phai ~0, kiem moi epoch

def shuf_idx(sb):
    '''Hoan vi TRONG LO sao cho moi hang nhan phim truoc cua BENH NHAN KHAC.

       Ban truoc dung `prior.roll(1, 0)`. Luc HUAN LUYEN thi dung (loader shuffle=True), nhung luc
       CHAM DIEM thi `predict` dung shuffle=False va `ORDER` sap theo
       ['subject_id','study_id','dicom_id'] => hang lien ke phan lon CUNG benh nhan.
       377,110/65,379 ~ 5.8 anh/benh nhan => ~83% so lan cuon 1 van ra CHINH nguoi do.
       => `prior_zero` khong con la doi chung, va moi `G_prior` doc duoc deu vo nghia.
       (43/44a khong dinh vi chung dung hoan vi tuong minh VA in ty le trung.)'''
    B = sb.shape[0]; idx = torch.arange(B, device=sb.device)
    p = (idx + 1) % B
    # Quet HET B buoc dich. Ban truoc chan cung o 32 va TRUOT tren du lieu that: `ORDER` sap theo
    # `subject_id`, mot benh nhan co the chiem >32 hang lien tiep (42 do co nguoi 157 study), nen
    # moi buoc dich <32 van roi vao chinh nguoi ay -> con 10.5% trung. Dry-run khong bat duoc vi
    # benh nhan gia chi co ~10 hang lien tiep.
    for k in range(2, B):
        bad = sb[p] == sb
        if not bad.any(): break
        p = torch.where(bad, (idx + k) % B, p)
    same = sb[p] == sb
    # Truong hop benh benh: CA LO cung mot benh nhan -> khong the tim doi chung. Khi do cat nhanh
    # (has=0) thay vi lay nham chinh benh nhan do. Bao cao ty le nay, khong giau.
    return p, same, float(same.float().mean())

# --- cong 12 (MOI, sau khi doi chung hong o lan chay truoc): hoan vi PHAI tranh cung benh nhan ---
# Kiem tren DUNG thu tu ORDER (sap theo subject_id) — do la truong hop XAU NHAT, va la truong hop
# `predict` that su gap vi no dung shuffle=False.
_sb_test = torch.from_numpy(SUB[:256].astype('int64')).to(dev)
_pi_t, _sm_t, _rt_t = shuf_idx(_sb_test)
print(f'cong 12a (THAM KHAO, truong hop XAU NHAT): tren thu tu ORDER nguyen ban -> '
      f'{_rt_t:.2%} trung benh nhan')
print('   `predict` gio SHUFFLE nen KHONG con gap truong hop nay; day chi la moc doi chieu.')
print('   (roll(1,0) ra ~83% · chan cung 32 buoc ra 10.5% · quet het B ra so tren)')
# Cong THAT: tren mot lo NGAU NHIEN — dung thu ma `predict` va vong lap huan luyen that su gap.
_rs12 = np.random.default_rng(12)
_rt_r = float(np.mean([shuf_idx(torch.from_numpy(
    SUB[_rs12.choice(len(SUB), BATCH, replace=False)].astype('int64')).to(dev))[2]
    for _ in range(20)]))
assert _rt_r < 0.01, f'lo NGAU NHIEN van {_rt_r:.2%} trung benh nhan -> doi chung khong dung duoc'
print(f'cong 12b (THAT): lo ngau nhien, {BATCH} mau x 20 lan -> {_rt_r:.3%} trung benh nhan OK')

def run_arms(M, E, patch, erow, need_w=False, I=None, prior=None, has=None, sb=None):
    out = {}
    pz = None
    for tag, cfg in ARMS.items():
        e = ehr_batch(M, E, tag, erow)
        p = patch
        if cfg.get('img'):
            if cfg.get('shuf'):
                if pz is None:
                    _pi, _sm, _rt = shuf_idx(sb); SHUF_SAME.append(_rt)
                    # hang khong tim duoc benh nhan khac -> cat nhanh, KHONG lay nham chinh no
                    pz = (prior[_pi], has[_pi] * (~_sm).float())
                pr, hs = pz
            else:
                pr, hs = prior, has
            p = I[tag](patch, pr, hs)
        out[tag] = M[tag](p, e, need_w=need_w)
    return out

@torch.no_grad()
def predict(M, E, I, idx, perm=None, bs=48):
    M.eval(); E.eval(); I.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    # SHUFFLE ca khi danh gia. Goc re cua loi doi chung: `ORDER` sap theo `subject_id`, nen voi
    # shuffle=False mot lo co the TOAN BO la mot benh nhan (mo phong: o B=48 co cua so 100%),
    # va luc do khong hoan vi nao trong lo tim duoc benh nhan khac. Shuffle xoa han van de.
    # Ghi du doan theo `pos` (chi so trong `idx`), khong theo thu tu duyet.
    for b in tqdm(loader(idx, eval_tf, True, bs, perm, nw=0, drop_last=False),
                  desc='eval', leave=False):
        pt = fwd(b['image'].to(dev, non_blocking=True))
        pp = fwd(b['image_prev'].to(dev, non_blocking=True)); hs = b['has'].to(dev)
        for t, (lg, _) in run_arms(M, E, pt, b['erow'], I=I, prior=pp, has=hs,
                                   sb=b['sub'].to(dev)).items():
            P[t][b['pos'].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    _bad = {t: int(np.isnan(v).sum()) for t, v in P.items() if np.isnan(v).any()}
    assert not _bad, f'predict cho ra NaN: {_bad}'
    for t in ARMS:
        assert np.abs(P[t]).sum(1).min() > 0, f'{t}: co hang khong duoc ghi -> `pos` sai'
    return P

def make_opt(M, E, I):
    assert not any(getattr(M[t], 'mode', None) == 'gate' for t in ARMS), '44b khong nen co arm gate'
    rest = [p for m in (M, E, I) for p in m.parameters() if p.requires_grad]
    dec = [p for p in rest if p.ndim > 1]; nod = [p for p in rest if p.ndim <= 1]
    return torch.optim.AdamW([{'params': dec, 'weight_decay': WD},
                              {'params': nod, 'weight_decay': 0.0}], lr=LR)

P_LOG = os.path.join(RES38C, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL  = loader(I_TF, train_tf, True, BATCH)

def finite_params(M, E):
    '''Tra ve arm dau tien co THAM SO khong huu han. Tham so hong thi moi forward sau deu NaN,
       nen phai bat o day chu khong phai o logit.'''
    for t in ARMS:
        for m in ([M[t]] + ([E[t]] if t in E else [])):
            for n_, p in m.named_parameters():
                if not torch.isfinite(p).all(): return t, n_
    return None, None

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK, IMG = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + ([ETK[t]] if t in ETK else [])
                                + ([IMG[t]] if t in IMG else []))
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    opt = make_opt(MOD, ETK, IMG)
    steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})

    # --- RESUME o muc EPOCH. Truoc day chi luu ckpt SAU KHI xong het epoch, nen mot lan
    #     dut o ep3 mat trang 3.5 gio. Gio moi epoch mot ckpt: dut thi mat DUNG mot epoch. ---
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt')
    ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk']); IMG.load_state_dict(_r['img'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        MOD.train(); ETK.train(); IMG.train()
        t0, tot, nb, n_bad = time.time(), 0.0, 0, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            pt = fwd(b['image'].to(dev, non_blocking=True)); y = b['y'].to(dev)
            pp = fwd(b['image_prev'].to(dev, non_blocking=True)); hs = b['has'].to(dev)
            loss, bad = 0.0, []
            for t, (lg, _) in run_arms(MOD, ETK, pt, b['erow'], I=IMG, prior=pp, has=hs,
                                       sb=b['sub'].to(dev)).items():
                if not torch.isfinite(lg).all():
                    bad.append(t); N_SKIP[t] += 1
                    continue                       # BO arm nay o buoc nay, KHONG giet ca lan chay
                loss = loss + crit(lg.float(), y)
            if bad:
                n_bad += 1
                if n_bad <= 3 or n_bad % 200 == 0:
                    _pt, _pn = finite_params(MOD, ETK)
                    _mx = {t: max(float(p.abs().max()) for p in PARAM_OF[t]) for t in bad}
                    print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad}')
                    print(f'     |tham so| lon nhat: ' + ' '.join(f'{t}:{v:.3g}' for t, v in _mx.items()))
                    print(f'     tham so hong: {_pt}/{_pn}' if _pt else
                          '     THAM SO van huu han -> tran o KICH HOAT, khong phai o trong so')
            if not torch.is_tensor(loss):
                opt.zero_grad(set_to_none=True); continue      # ca 12 arm deu hong o buoc nay
            (loss / ACCUM).backward(); tot += float(loss.detach()); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc. clip_grad_norm_ voi total_norm = nan cho ra
                # clip_coef = nan va BOI BAN toan bo tham so — mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()   # arm nay khong buoc
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP:
                        N_CLIP[t] += 1
                N_STEP += 1
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        PV = predict(MOD, ETK, IMG, I_VA); PS = predict(MOD, ETK, IMG, I_SL)
        ensure_res()          # mot epoch ~25 phut khong ghi gi -> FUSE co the da danh roi thu muc
        for t, p in PV.items(): np.save(os.path.join(RES38C, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        _e = {}
        for t, p in PV.items():
            a = score(p)[0]
            _e[t] = dict(mAP_anh=float(np.nanmean(a)), **tiers(a),
                         sel=float(np.nanmean(ap_cols(YM[SL], PS[t]))))
        LOG[sk][str(ep)] = _e; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                        opt=opt.state_dict(), sch=sch.state_dict(), n_clip=N_CLIP,
                        n_skip=N_SKIP, n_step=N_STEP, ep=ep, seed=seed), P_RES)
        print(f'{sk} ep{ep+1:>2} loss {tot/max(nb,1):.4f} | {len(ARMS)} arm '
              f'| {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | {(time.time()-t0)/60:.1f} ph')
        _ss = float(np.mean(SHUF_SAME)) if SHUF_SAME else 0.0
        print(f'   doi chung `prior_zero`: {_ss:.3%} truong hop TRUNG benh nhan '
              f'(43/44a do 0.01%; ban `roll(1,0)` cu ra ~83% luc eval)')
        assert _ss < 0.01, (f'doi chung TRUNG benh nhan {_ss:.1%} -> `prior_zero` KHONG phai doi '
                            'chung, moi `G_prior` vo nghia. (Hang trung da bi cat has=0, nhung ty '
                            'le cao nghia la doi chung mat qua nhieu mau.)')
        SHUF_SAME.clear()
        _gp = _e['prior_d']['mAP_anh'] - _e['prior_zero']['mAP_anh']
        print(f'   G_prior = prior_d - prior_zero = {_gp:+.4f}   <- CON SO CAN DO'
              f'   (nguong MUON tu 44a: 0.0013)')
        # In DU CA 12 ARM. Dong rut gon truoc day khien nguoi doc tuong chi co 4 arm dang chay.
        _nm = [t.replace('mldec_', '') or 'mldec' for t in ARMS]
        print('   VAL      ' + ' '.join(f'{n:>9}:{_e[t]["mAP_anh"]:.4f}' for n, t in zip(_nm, ARMS)))
        print('   LAT CHON ' + ' '.join(f'{n:>9}:{_e[t]["sel"]:.4f}' for n, t in zip(_nm, ARMS)))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        _f = {t: N_CLIP[t]/max(N_STEP,1) for t in ARMS}
        if max(_f.values()) > 0.01:
            print('   clip kich hoat: ' + ' '.join(f'{t}:{v:.0%}' for t, v in _f.items() if v > 0.01))
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO (khong huu han): ' +
                  ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0))
            print('   -> arm nao bi bo nhieu thi con so cua no KHONG doc duoc; ghi vao REPORT.')
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                    seed=seed, n_skip=N_SKIP), os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; LOG[sk]['n_skip'] = N_SKIP; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong' + (f' | buoc bi bo: {N_SKIP}' if max(N_SKIP.values()) else ''))
print('tat ca seed xong')

cong 12a (THAM KHAO, truong hop XAU NHAT): tren thu tu ORDER nguyen ban -> 0.00% trung benh nhan
   `predict` gio SHUFFLE nen KHONG con gap truong hop nay; day chi la moc doi chieu.
   (roll(1,0) ra ~83% · chan cung 32 buoc ra 10.5% · quet het B ra so tren)
cong 12b (THAT): lo ngau nhien, 64 mau x 20 lan -> 0.000% trung benh nhan OK


s42 ep1/6:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep 1 loss 0.2542 | 6 arm | 70 img/s | 52.3 ph
   doi chung `prior_zero`: 0.000% truong hop TRUNG benh nhan (43/44a do 0.01%; ban `roll(1,0)` cu ra ~83% luc eval)
   G_prior = prior_d - prior_zero = +0.0031   <- CON SO CAN DO   (nguong MUON tu 44a: 0.0013)
   VAL          mldec:0.2396   prior_a:0.2428   prior_d:0.2412 prior_zero:0.2381        t1:0.2397  t1_prior:0.2417
   LAT CHON     mldec:0.2319   prior_a:0.2380   prior_d:0.2360 prior_zero:0.2328        t1:0.2352  t1_prior:0.2359
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`


...

s42 ep 5 loss 0.2202 | 6 arm | 70 img/s | 52.1 ph
   doi chung `prior_zero`: 0.000% truong hop TRUNG benh nhan (43/44a do 0.01%; ban `roll(1,0)` cu ra ~83% luc eval)
   G_prior = prior_d - prior_zero = +0.0015   <- CON SO CAN DO   (nguong MUON tu 44a: 0.0013)
   VAL          mldec:0.2461   prior_a:0.2480   prior_d:0.2469 prior_zero:0.2454        t1:0.2517  t1_prior:0.2559
   LAT CHON     mldec:0.2388   prior_a:0.2428   prior_d:0.2428 prior_zero:0.2408        t1:0.2438  t1_prior:0.2477
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`


s42 ep6/6:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep 6 loss 0.2164 | 6 arm | 70 img/s | 52.1 ph
   doi chung `prior_zero`: 0.000% truong hop TRUNG benh nhan (43/44a do 0.01%; ban `roll(1,0)` cu ra ~83% luc eval)
   G_prior = prior_d - prior_zero = -0.0010   <- CON SO CAN DO   (nguong MUON tu 44a: 0.0013)
   VAL          mldec:0.2462   prior_a:0.2472   prior_d:0.2446 prior_zero:0.2455        t1:0.2513  t1_prior:0.2562
   LAT CHON     mldec:0.2385   prior_a:0.2418   prior_d:0.2398 prior_zero:0.2409        t1:0.2440  t1_prior:0.2478
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
s42: xong
tat ca seed xong


## C7 · Bảng + **biên độ ghép cặp**

Chỉ `arm − mldec` được đọc: các arm dùng chung một forward, chung thứ tự dữ liệu, chung phép tăng
cường. Một hiệu chỉ đáng tin khi **cùng dấu qua mọi seed** VÀ `|d| > biên độ`.

In [ ]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
rows = [dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
        for t, v in LOG[f's{s}'][EPk].items()]
R = pd.DataFrame(rows); assert len(R), 'chua co ket qua nao'
for s in SEEDS:
    if EPk not in LOG.get(f's{s}', {}): print(f's{s}: chua co ep{EPOCHS} — bien do chua doc duoc')

print('=== TUYET DOI (TB qua seed) ===')
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'],
                    aggfunc='mean').reindex(list(ARMS))
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(list(ARMS))
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))

print('\n=== HIEU GHEP CAP  arm - mldec  (day moi la cho duoc doc) ===')
base = {s: R[(R.arm=='mldec') & (R.seed==s)] for s in SEEDS}
D = []
for t in ARMS:
    if t == 'mldec': continue
    d, dt = [], []
    for s in SEEDS:
        r = R[(R.arm==t) & (R.seed==s)]
        if len(r) and len(base[s]):
            d.append(float(r.mAP_anh.iloc[0] - base[s].mAP_anh.iloc[0]))
            dt.append(float(r['tail'].iloc[0] - base[s]['tail'].iloc[0]))
    if d: D.append(dict(arm=t, d_anh=np.mean(d), bien_do=max(d)-min(d),
                        cung_dau=all(x > 0 for x in d) or all(x < 0 for x in d),
                        d_tail=np.mean(dt), bien_do_tail=max(dt)-min(dt)))
D = pd.DataFrame(D)
print(D.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))

print('\n=== G_ehr = gate - gate_zero  (doi chung KHOP THAM SO, ke ca MicroBag) ===')
for a, b in (('prior_d','prior_zero'),):
    g  = [float(R[(R.arm==a)&(R.seed==s)].mAP_anh.iloc[0] - R[(R.arm==b)&(R.seed==s)].mAP_anh.iloc[0])
          for s in SEEDS if len(R[(R.arm==a)&(R.seed==s)])]
    gt = [float(R[(R.arm==a)&(R.seed==s)]['tail'].iloc[0] - R[(R.arm==b)&(R.seed==s)]['tail'].iloc[0])
          for s in SEEDS if len(R[(R.arm==a)&(R.seed==s)])]
    if g: print(f'  {a:<12} - {b:<16} {np.mean(g):+.4f} (bien do {max(g)-min(g):.4f}) | '
                f'tail {np.mean(gt):+.4f} (bien do {max(gt)-min(gt):.4f})')
ensure_res()
R.to_csv(os.path.join(RES38C, 'main_table.csv'), index=False)
D.to_csv(os.path.join(RES38C, 'paired_diff.csv'), index=False)
print(f'\n  MOC de doi chieu (38b, one-hot 30, backbone `ft`, E=8 epoch cuoi): mldec 0.2403')

=== TUYET DOI (TB qua seed) ===
             head  mAP_anh  medium    sel   tail  bien_do
arm                                                      
mldec      0.4573   0.2462  0.1748 0.2385 0.1008   0.0000
prior_a    0.4617   0.2472  0.1778 0.2418 0.0944   0.0000
prior_d    0.4591   0.2446  0.1786 0.2398 0.0870   0.0000
prior_zero 0.4578   0.2455  0.1756 0.2409 0.0965   0.0000
mldec_t1   0.4642   0.2513  0.1784 0.2440 0.1058   0.0000
t1_prior   0.4707   0.2562  0.1838 0.2478 0.1080   0.0000

=== HIEU GHEP CAP  arm - mldec  (day moi la cho duoc doc) ===
       arm   d_anh  bien_do  cung_dau  d_tail  bien_do_tail
   prior_a +0.0009  +0.0000      True -0.0063       +0.0000
   prior_d -0.0017  +0.0000      True -0.0138       +0.0000
prior_zero -0.0007  +0.0000      True -0.0043       +0.0000
  mldec_t1 +0.0051  +0.0000      True +0.0050       +0.0000
  t1_prior +0.0099  +0.0000      True +0.0072       +0.0000

=== G_ehr = gate - gate_zero  (doi chung KHOP THAM SO, ke ca MicroBag) ===
  pri

## C8 · Nhóm nhãn **nhiễm trùng** — nơi từ vựng đầy đủ phải hiện ra

Vi khuẩn bị `top-30` cắt là tác nhân **viêm phổi**. Nếu danh tính đầy đủ có giá trị thì nó phải nhấc
đúng nhóm này, **không phải macro** — 40 nhãn gộp lại sẽ pha loãng mọi hiệu ứng thật xuống dưới sàn.
Nhóm lấy từ `33` C7, dùng làm **kiểm chứng**, không hard-code vào kiến trúc.

In [ ]:
INF = [l for l in LAB40 if re.search(r'pneumon|tubercul|infect|consolidat|abscess|empyema', l, re.I)]
IDX_INF = np.array([LAB40.index(l) for l in INF])
# NHOM CAU TRUC — day moi la nhom cua PHIM TRUOC. 44a do: Hydropneumothorax +0.0415 ·
# Pneumothorax +0.0291 · Hernia +0.0287 · Mass +0.0265 · Rib Fracture +0.0188 · Fracture +0.0159
STRUCT = [l for l in LAB40 if re.search(
    r'pneumothorax|hernia|fracture|mass|nodule|calcif|scoliosis|support device', l, re.I)]
IDX_STR = np.array([LAB40.index(l) for l in STRUCT], dtype=int)
print(f'nhom CAU TRUC ({len(STRUCT)} nhan): {STRUCT}')
print(f'nhom nhiem trung ({len(INF)} nhan): {INF}\n')
sk = f's{SEEDS[0]}'
_res = []
for t in (ARMS if len(IDX_INF) else []):
    f = os.path.join(RES38C, f'p_{t}_{sk}_ep{EPOCHS}.npy')
    if not os.path.exists(f): continue
    a = ap_cols(YM[VA], np.load(f))
    _res.append(dict(arm=t, macro=float(np.nanmean(a)),
                     cau_truc=float(np.nanmean(a[IDX_STR])) if len(IDX_STR) else np.nan,
                     nhiem_trung=float(np.nanmean(a[IDX_INF])) if len(IDX_INF) else np.nan,
                     tail=float(np.nanmean(a[TIER['tail']]))))
T8 = pd.DataFrame(_res)
T8 = T8.set_index('arm') if len(T8) else T8
if len(T8) and 'mldec' in T8.index:
    for c in ('macro', 'cau_truc', 'nhiem_trung', 'tail'): T8[f'd_{c}'] = T8[c] - T8.loc['mldec', c]
print(T8.to_string(float_format=lambda v: f'{v:+.4f}') if len(T8) else '  (khong co du lieu)')
print('\n  DOC: PHIM TRUOC an nhom CAU TRUC (44a: tran khi/thoat vi/gay xuong), EHR an nhom')
print('       NHIEM TRUNG (38c: col_cat +0.0186). Hai nguon HAI NHOM khac nhau -> `t1_prior`')
print('       cho biet chung CONG DUOC hay CHONG LAN.')
print('  VA: `prior_zero` cung tap trung o nhom nao thi phan do KHONG den tu phim truoc')
print('      (38d da bat dung loi nay: t1_zero khong co EHR ma van tap trung 3.8x o nhiem trung).')

nhom CAU TRUC (10 nhan): ['Calcification of the Aorta', 'Clavicle Fracture', 'Fracture', 'Hernia', 'Hydropneumothorax', 'Mass', 'Nodule', 'Pneumothorax', 'Rib Fracture', 'Support Devices']
nhom nhiem trung (3 nhan): ['Consolidation', 'Pneumonia', 'Tuberculosis']

             macro  cau_truc  nhiem_trung    tail  d_macro  d_cau_truc  d_nhiem_trung  d_tail
arm                                                                                          
mldec      +0.2462   +0.3227      +0.1994 +0.1008  +0.0000     +0.0000        +0.0000 +0.0000
prior_a    +0.2472   +0.3333      +0.2073 +0.0944  +0.0009     +0.0106        +0.0079 -0.0063
prior_d    +0.2446   +0.3301      +0.2103 +0.0870  -0.0017     +0.0074        +0.0109 -0.0138
prior_zero +0.2455   +0.3202      +0.2015 +0.0965  -0.0007     -0.0025        +0.0021 -0.0043
mldec_t1   +0.2513   +0.3389      +0.2110 +0.1058  +0.0051     +0.0161        +0.0116 +0.0050
t1_prior   +0.2562   +0.3450      +0.2161 +0.1080  +0.0099     +0.0223        

## C9 · Tráo nhân quả — hoán vị **cả** ma trận EHR **lẫn** túi vi khuẩn

In [ ]:
ck = torch.load(os.path.join(CKPT, f's{SEEDS[0]}.pt'), map_location=dev, weights_only=False)
MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk']); IMG.load_state_dict(ck['img'])
rs = np.random.default_rng(0); perm = rs.permutation(len(I_VA))
P0 = predict(MOD, ETK, IMG, I_VA); P1 = predict(MOD, ETK, IMG, I_VA, perm=I_VA[perm])
lg = lambda p: np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
CONST = [t for t, c in ARMS.items() if c.get('const')]
print(f'=== TRAO NHAN QUA (s{SEEDS[0]}) — EHR + tui vi khuan sang benh nhan khac, NHAN giu nguyen ===')
print('  Doc HAI cot cung luc: corr mot minh QUA CHAT — logit gan nhu khong doi ma THU TU doi,')
print('  ma AP la chi so XEP HANG.')
for t in ARMS:
    if not ARMS[t]['ehr']: continue
    r = float(np.corrcoef(lg(P0[t]).ravel(), lg(P1[t]).ravel())[0, 1])
    d = float(np.nanmean(ap_cols(YM[VA], P1[t])) - np.nanmean(ap_cols(YM[VA], P0[t])))
    v = ('EHR co tac dong that' if d <= -0.002 else
         'EHR KHONG duoc dung'  if r > 0.9999 and d > -0.0005 else 'vung xam')
    print(f'  {t:<18} corr(logit) {r:.6f} | mAP khi trao {d:+.4f}   {v}')
    if t in CONST:
        assert r > 1 - 1e-9 and abs(d) < 1e-9, \
            f'{t} la doi chung HANG SO nhung trao van doi ket qua (corr {r:.9f}, d {d:+.6f}) -> ro ri'
print(f'\n  moc 06b: corr = 0.999934 (H6)')
print(f'  cong: doi chung hang so {CONST} bat bien duoi phep trao')

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

=== TRAO NHAN QUA (s42) — EHR + tui vi khuan sang benh nhan khac, NHAN giu nguyen ===
  Doc HAI cot cung luc: corr mot minh QUA CHAT — logit gan nhu khong doi ma THU TU doi,
  ma AP la chi so XEP HANG.
  mldec_t1           corr(logit) 0.988930 | mAP khi trao -0.0097   EHR co tac dong that
  t1_prior           corr(logit) 0.992150 | mAP khi trao -0.0067   EHR co tac dong that

  moc 06b: corr = 0.999934 (H6)
  cong: doi chung hang so [] bat bien duoi phep trao


## C10 · Ghi `REPORT_STAGE4_NB44B.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb44b` — phim chụp TRƯỚC ở mức patch (1369 token)'); w('')
w(f'`{RUN_TAG}` · backbone `ft` = `38a/{BB_RUN}` · `asl40` · `E={EPOCHS}` · **epoch cuối** · '
  f'seeds {list(SEEDS)} · `V={V_MIC-1}` vi khuẩn (so với 30 của `29b`) · val {VA.sum():,} ảnh.'); w('')
w('## `top-30` đã vứt đi gì (`29c`)'); w('')
w('```'); w('do phu LAN MOC              84.7%')
w('study MAT SACH danh tinh     7.4%   (4,675 / 63,293)')
w('study mat MOT PHAN          13.4%   (8,500 / 63,293)')
w('bi cat: HAEMOPHILUS INFLUENZAE 336 · MORAXELLA CATARRHALIS 302 · INFLUENZA A 238')
w('        ASPERGILLUS FUMIGATUS 235 · ACINETOBACTER BAUMANNII 215'); w('```'); w('')
w('| arm | mAP ảnh | biên độ | tail | nhiễm trùng | `arm − mldec` | biên độ | cùng dấu |')
w('|---|---|---|---|---|---|---|---|')
for t in ARMS:
    r = R[R.arm == t]
    if not len(r): continue
    d = D[D.arm == t]
    _inf = (f'{T8.loc[t, "nhiem_trung"]:.4f}'
            if 'T8' in globals() and t in T8.index else '—')
    w(f'| `{t}` | {r.mAP_anh.mean():.4f} | {r.mAP_anh.max()-r.mAP_anh.min():.4f} | '
      f'{r["tail"].mean():.4f} | {_inf} | ' +
      (f'{d.d_anh.iloc[0]:+.4f} | {d.bien_do.iloc[0]:.4f} | '
       f'{"co" if d.cung_dau.iloc[0] else "KHONG"} |' if len(d) else '— | — | — |'))
w('')
w('Mốc `38b` (one-hot 30, cùng backbone, `E=8`, epoch cuối): `mldec 0.2403`.'); w('')
w('**Chỉ `arm − mldec` được đọc** — các arm dùng chung một forward nên hiệu của chúng là **ghép '
  'cặp**. Một hiệu đáng tin khi **cùng dấu qua mọi seed** VÀ `|d| > biên độ`.'); w('')
w('## Những gì kết quả này KHÔNG loại trừ'); w('')
w('- **Biểu diễn vẫn còn chỗ hở.** `EHRColTok` phát token cho **mọi** cột, kể cả cột vắng mặt '
  '(`T_j = b_j`). `B2b` của kế hoạch nói *"thiếu: KHÔNG sinh token"* — chưa làm, và chưa đo so với '
  'phương án kia.')
w('- **Comorbidity** vẫn là cờ one-hot, cùng dạng bệnh lý với vi khuẩn top-30. Cùng cách sửa.')
w('- **Backbone đông cứng** ⇒ EHR không thể định hình biểu diễn, chỉ điều biến đầu ra (`39`, B2d).')
w(f'- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa. Không con số nào ở đây nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện.')
TXT = '\n'.join(L)
ensure_res()
open(os.path.join(RES38C, 'REPORT_STAGE4_NB44B.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REPORT · `nb44b` — phim chụp TRƯỚC ở mức patch (1369 token)

`b2_prior_patch_fix` · backbone `ft` = `38a/a2_E3` · `asl40` · `E=6` · **epoch cuối** · seeds [42] · `V=305` vi khuẩn (so với 30 của `29b`) · val 39,293 ảnh.

## `top-30` đã vứt đi gì (`29c`)

```
do phu LAN MOC              84.7%
study MAT SACH danh tinh     7.4%   (4,675 / 63,293)
study mat MOT PHAN          13.4%   (8,500 / 63,293)
bi cat: HAEMOPHILUS INFLUENZAE 336 · MORAXELLA CATARRHALIS 302 · INFLUENZA A 238
        ASPERGILLUS FUMIGATUS 235 · ACINETOBACTER BAUMANNII 215
```

| arm | mAP ảnh | biên độ | tail | nhiễm trùng | `arm − mldec` | biên độ | cùng dấu |
|---|---|---|---|---|---|---|---|
| `mldec` | 0.2462 | 0.0000 | 0.1008 | 0.1994 | — | — | — |
| `prior_a` | 0.2472 | 0.0000 | 0.0944 | 0.2073 | +0.0009 | 0.0000 | co |
| `prior_d` | 0.2446 | 0.0000 | 0.0870 | 0.2103 | -0.0017 | 0.0000 | co |
| `prior_zero` | 0.2455 | 0.0000 | 0.0965 | 0.2015 | -0.0007 | 0.0000 | co |
| `mldec_t1` | 0.2513 | 0.0000 | 0.1058 | 0.2